# Graph, contratos y resultados · 2.1.4

Un Agent organiza una capacidad; un System coordina la aplicación; Environment procesa episodios. Solo Graph y compile, sin check. Python determinista, sin inferencia.

In [2]:
"""Graph composition: typed Tools, two Agents, episodes and isolated evaluation."""
import pydantic
import agentic_systems as toolkit


class Amount(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(strict=True, extra="forbid")
    amount: int = pydantic.Field(gt=0)


class Fees(Amount):
    fee: int = pydantic.Field(ge=0)


class Quote(Fees):
    total: int = pydantic.Field(gt=0)


class Decision(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(strict=True, extra="forbid")
    decision: str = pydantic.Field(pattern="^(approve|review)$")


class PaymentState(Amount):
    fee: int | None = None
    total: int | None = None
    decision: str | None = None
    status: str | None = None


class PaymentOutput(Quote):
    decision: str = pydantic.Field(pattern="^(approve|review)$")
    status: str = pydantic.Field(pattern="^(approved|pending_review)$")


class Policy(pydantic.BaseModel):
    model_config = pydantic.ConfigDict(strict=True, extra="forbid")
    fee_bps: int = pydantic.Field(ge=0)
    ceiling: int = pydantic.Field(gt=0)


@toolkit.tool(input_schema=Amount, output_schema=Fees)
def calculate_fee(payment: Amount, *, context: toolkit.Context) -> Fees:
    return Fees(amount=payment.amount, fee=payment.amount * context.data["fee_bps"] // 10000)


@toolkit.tool(input_schema=Fees, output_schema=Quote)
def calculate_total(payment: Fees) -> Quote:
    return Quote(amount=payment.amount, fee=payment.fee, total=payment.amount + payment.fee)


@toolkit.tool(input_schema=Quote, output_schema=Decision)
def decide(quote: Quote, *, context: toolkit.Context) -> Decision:
    return Decision(decision="approve" if quote.total <= context.data["ceiling"] else "review")


capabilities = toolkit.skill(
    name="fees", tools=(calculate_fee, calculate_total),
    instructions="Calculate fees and totals in integer cents from the declared policy.",
)



## Agent y Tools Pydantic

El grafo calcula comisión y total. input_schema entrega un modelo al único parámetro de negocio; las Tools validan entradas y salidas.

In [3]:
# Agent graph: organize the steps of one capability.
fee_graph = toolkit.Graph(PaymentState, input_schema=Amount, output_schema=Quote)
fee_graph.add_node("fee", calculate_fee)
fee_graph.add_node("total", calculate_total)
fee_graph.add_edge(toolkit.START, "fee")
fee_graph.add_edge("fee", "total")
fee_graph.add_edge("total", toolkit.END)
fee_agent = toolkit.Agent("fee_capability", graph=fee_graph, skills=(capabilities,))

reviewer = toolkit.Agent(
    "reviewer", runtime=toolkit.runtime(provider="python-runtime"),
    skills=(toolkit.skill(name="review", tools=(decide,)),),
)


def fee_step(state):
    return fee_agent.run({"amount": state.amount})


def review_step(state):
    return reviewer.run({"amount": state.amount, "fee": state.fee, "total": state.total})


def choose(state):
    return state.decision


def approve(state):
    return {"status": "approved"}


def require_review(state):
    return {"status": "pending_review"}




## System: dos Agents y una decisión

Las funciones muestran la proyección de entrada. Context contiene la política validada; Memory conserva episodios.

In [4]:
# System graph: coordinate Agents and the application decision.
system_graph = toolkit.Graph(PaymentState, input_schema=Amount, output_schema=PaymentOutput)
system_graph.add_node("fees", fee_step)
system_graph.add_node("review", review_step)
system_graph.add_node("approve", approve)
system_graph.add_node("review_required", require_review)
system_graph.add_edge(toolkit.START, "fees")
system_graph.add_edge("fees", "review")
system_graph.add_conditional_edges("review", choose, {"approve": "approve", "review": "review_required"})
system_graph.add_edge("approve", toolkit.END)
system_graph.add_edge("review_required", toolkit.END)

memory = toolkit.Memory()
system = toolkit.System(
    "payments", graph=system_graph,
    context=toolkit.Context(Policy(fee_bps=100, ceiling=10000), sources=("payment-policy-v1",)),
    memory=memory,
)
harness = system.compile()




## Environment y Eval

La recompensa es una métrica explícita calculada en update, registrada en state/history; no hay reward= ni aprendizaje automático. El oráculo expected_status no se entrega al System. Cada caso de Eval usa una vista de Memory aislada.

In [5]:
# Environment graph: episodes, state and an explicit business reward metric.
def observe_payment(record, state):
    return {"amount": record["amount"]}


def update_day(state, record, result):
    reward = 1 if result.ok and result.data["status"] == record["expected_status"] else 0
    return {"processed": state["processed"] + 1, "reward": reward, "total_reward": state["total_reward"] + reward}


episode_graph = toolkit.Graph(PaymentState, input_schema=Amount, output_schema=PaymentOutput)
episode_graph.add_node("payment", system)
episode_graph.add_edge(toolkit.START, "payment")
episode_graph.add_edge("payment", toolkit.END)
environment = toolkit.Environment(
    "daily_payments", graph=episode_graph,
    records=({"amount": 5000, "expected_status": "approved"},
             {"amount": 12000, "expected_status": "pending_review"}),
    state={"processed": 0, "reward": 0, "total_reward": 0},
    observe=observe_payment, update=update_day,
)


def check_quote(result, case):
    amount = case["input"]["amount"]
    fee = amount // 100
    decision = "approve" if amount + fee <= 10000 else "review"
    status = "approved" if decision == "approve" else "pending_review"
    return {"ok": result.data == {"amount": amount, "fee": fee, "total": amount + fee,
                                  "decision": decision, "status": status}}


# Eval automatically gives each case its own Memory view, without changing this API.
evaluation = toolkit.Eval(
    "payment_acceptance", unit=system,
    cases=({"name": "approved", "input": {"amount": 5000}, "expected": {"ok": True}},
           {"name": "review", "input": {"amount": 12000}, "expected": {"ok": True}}),
    checks=(check_quote,),
)



In [6]:
result = harness.run({"amount": 5000}, timeout_s=30)
episode = environment.run(timeout_s=30)
before_eval = memory.read()
checked = evaluation.run(timeout_s=30)
assert result.ok and episode.ok and checked.ok
assert checked.data["passed"] == 2
assert memory.read() == before_eval
assert episode.data["summary"]["state"]["total_reward"] == 2
assert all(output.check_invariants().ok for output in (result, episode, checked))
for output in (result, episode, checked):
    print(output.human_results())
print("Production Memory episodes:", len(memory.read()))
print("Reward metric:", episode.data["summary"]["state"])

payments: COMPLETED
Execution: run-e8cb534f9f60410d9c1a8d738707ea60
Input: {"amount":5000}
Output: {"amount":5000,"fee":50,"total":5050,"decision":"approve","status":"approved"}
Process:
- Agent fee_capability: completed
- Tool calculate_fee: completed -> {"amount":5000,"fee":50}
- Tool calculate_total: completed -> {"amount":5000,"fee":50,"total":5050}
- Agent reviewer: completed
- Tool decide: completed -> {"decision":"approve"}
- Decision review -> "approve"
- Step approve: completed
daily_payments: COMPLETED
Execution: run-c09b5b60b3e048acaa25af8e7c1b447d
Input: {"records_observed":[{"amount":5000,"expected_status":"approved"},{"amount":12000,"expected_status":"pending_review"}]}
Output: {"outputs":[{"amount":5000,"fee":50,"total":5050,"decision":"approve","status":"approved"},{"amount":12000,"fee":120,"total":12120,"decision":"review","status":"pending_review"}],"summary":{"state":{"processed":2,"reward":1,"total_reward":2},"processed":2,"terminated":true,"truncated":false,"reason

Input 5000 → comisión 50 → total 5050 → aprobado. Environment también procesa 12000 → comisión 120 → total 12120 → revisión. Eval comprueba ambos resultados sin escribir en la Memory de uso normal. compile prepara sin inferencia; human_results resume trabajo observado y conserva el detalle en to_dict/lineage.